In [1]:
import numpy as np
import itertools
import math

np.random.seed(12321)

# Pfaffians

In [2]:
class Pfaffian:
    def __init__(self, data):
        self.data = data.copy()
        self.inv  = np.linalg.inv(self.data)
        self.delta = np.zeros(self.data.shape[0])
        self.active = -1
       
    def rowPivoting(self, tmp, i):
        #taken from Bajdich thesis
        n = tmp.shape[0]
        if n%2 != 0:
            return 0
            
        TINY = 1.0e-20
        backup = np.zeros(n)
        d = 1
        k = 0
        big = 0.0
        for j in range(i+1, n):
            temp = np.abs(tmp[i,j])
            if (temp > big):
                big = temp
                k = j
        if big < TINY:
            print("Singular row in matrix!!!")
            tmp[i,i+1] = TINY
        if k!=(i+1):
            for j in range(i, n):
                backup[j] = tmp[j,i+1]
                tmp[j, i+1] = tmp[j,k]
                tmp[j,k]= backup[j]
            for j in range(i, n):
                backup[j] = tmp[i+1,j]
                tmp[i+1,j] = tmp[k,j]
                tmp[k,j] = backup[j]
            d *= -1
        return d

    def evaluate(self):
        #taken from Bajdich thesis
        n = self.data.shape[0]
        if n%2 != 0:
            return 0
        tmp = self.data.copy()
        pf = 1.0
        d = 1
        for i in range(0, n, 2):
            d *= self.rowPivoting(tmp, i)
    
            for j in range(i+2, n):
                fac = -tmp[i,j] / tmp[i,i+1]
                for k in range(i+1, n):
                    tmp[k,j] += fac * tmp[k, i+1]
                    tmp[j,k] += fac * tmp[i+1, k]
            pf *= tmp[i, i+1]
        return pf * d

    def perm_sign(self, perm):
        """
        Compute the sign (+1 or -1) of a permutation by counting inversions.
        perm is a tuple/list of integers 0..N-1 in some order.
        """
        sign = 1
        N = len(perm)
        for i in range(N):
            for j in range(i+1, N):
                if perm[i] > perm[j]:
                    sign = -sign
        return sign

    def evaluate_bruteforce(self):
        """
        Brute‐force Pfaffian of a 2n×2n skew‐symmetric matrix A (as a nested list or array).
        Returns a floating‐point result.
        """
        A = self.data
        N = len(A)
        if N % 2 != 0:
            raise ValueError("Pfaffian is only defined for even‐dimension matrices")
        n = N // 2
    
        total = 0.0
        for sigma in itertools.permutations(range(N)):
            s = self.perm_sign(sigma)
            prod = 1.0
            # form the product of A[sigma[2i], sigma[2i+1]] for i=0..n-1
            for i in range(n):
                u = sigma[2*i]
                v = sigma[2*i + 1]
                prod *= A[u][v]
            total += s * prod
    
        # divide by 2^n * n!
        return total / (2**n * math.factorial(n))

    def ratio(self, proposed, i):
        self.active = i
        self.delta = proposed[:] - self.data[i,:]
        return 1.0 + self.delta.dot(self.inv[:,i])

    def updateInverse(self):
        """
        Update the inverse of the matrix using the rank-2 update formula.
        
        We define:
            u = e_i        (the i-th standard basis vector)
            v = proposed - self.data[i,:]
        and write:
            B = A + u vᵀ - v uᵀ.
        
        Then, using the Sherman–Morrison–Woodbury formula for a rank-2 update:
            B⁻¹ = A⁻¹ - A⁻¹ U (I_2 + V A⁻¹ U)⁻¹ V A⁻¹,
        where U = [u, v] and V = [vᵀ; -uᵀ].
        """
        n = self.data.shape[0]
        
        # Define u = e_i
        u = np.zeros(n)
        u[self.active] = 1.0
        
        # Define v as the change in row i
        #v = proposed - self.data[i, :]
        # v = self.delta

        # Build U = [u, v] as an n x 2 matrix.
        U = np.column_stack((u, self.delta))
        
        # Build V = [v^T, -u^T] as a 2 x n matrix.
        V = np.vstack((self.delta, -u))
        
        # Compute the 2 x 2 matrix M = V @ A⁻¹ @ U.
        M = V @ self.inv @ U
        
        # Invert the small 2 x 2 matrix I + M.
        inv_factor = np.linalg.inv(np.eye(2) + M)
        
        # Update the inverse using the SMW formula.
        self.inv = self.inv - self.inv @ U @ inv_factor @ V @ self.inv
                
    def accept(self):
        assert self.active >= 0
        self.data[self.active, :] += self.delta
        self.data[:, self.active] = -self.data[self.active,:]
        self.updateInverse()
        self.active = -1
        

In [3]:
N = 3
A = np.random.random((2*N, 2*N))
A = 0.5 * (A - A.T)

B = A.copy()
update = np.random.random(2*N)
ie = 4
update[ie] = 0
B[ie, :] = update[:].copy()
B[:, ie] = -update[:].copy()
print(A)
print(B)


print(np.linalg.inv(A))

print(update)

[[ 0.          0.18250294 -0.20363673 -0.25720828 -0.02146717  0.06856466]
 [-0.18250294  0.          0.2460778   0.0209037   0.10986746  0.3617668 ]
 [ 0.20363673 -0.2460778   0.          0.07461913 -0.01233308 -0.12637421]
 [ 0.25720828 -0.0209037  -0.07461913  0.         -0.32933761 -0.09674686]
 [ 0.02146717 -0.10986746  0.01233308  0.32933761  0.          0.06171134]
 [-0.06856466 -0.3617668   0.12637421  0.09674686 -0.06171134  0.        ]]
[[ 0.          0.18250294 -0.20363673 -0.25720828 -0.56637198  0.06856466]
 [-0.18250294  0.          0.2460778   0.0209037  -0.69536227  0.3617668 ]
 [ 0.20363673 -0.2460778   0.          0.07461913 -0.3739933  -0.12637421]
 [ 0.25720828 -0.0209037  -0.07461913  0.         -0.69729468 -0.09674686]
 [ 0.56637198  0.69536227  0.3739933   0.69729468 -0.          0.24455721]
 [-0.06856466 -0.3617668   0.12637421  0.09674686 -0.24455721  0.        ]]
[[-2.71503989e-16  1.81595609e+00  4.32396207e+00  9.92372604e-01
  -2.35068908e-01 -2.92715884e+0

In [4]:
pfA = Pfaffian(A)
pfB = Pfaffian(B)

vA = pfA.evaluate()
vbfA = pfA.evaluate_bruteforce()
vB = pfB.evaluate()
print(pfB.evaluate_bruteforce())

-0.020779936550488032


In [5]:
print(vA,vbfA)

-0.024797647365574372 -0.024797647365574358


In [6]:
newB = vA * pfA.ratio(update, ie)
print(newB, vB)

pfA.accept()

print("max inverse error: {}".format(np.max(pfA.inv - np.linalg.inv(B))))

-0.020779936550488032 -0.020779936550488008
max inverse error: 7.105427357601002e-15


In [7]:
C = B.copy()
update = np.random.random(2*N)
ie = 1
update[ie] = 0
C[ie, :] = update[:].copy()
C[:, ie] = -update[:].copy()
print(C)

[[ 0.         -0.80300995 -0.20363673 -0.25720828 -0.56637198  0.06856466]
 [ 0.80300995 -0.          0.72633347  0.98201802  0.7751635   0.66480625]
 [ 0.20363673 -0.72633347  0.          0.07461913 -0.3739933  -0.12637421]
 [ 0.25720828 -0.98201802 -0.07461913  0.         -0.69729468 -0.09674686]
 [ 0.56637198 -0.7751635   0.3739933   0.69729468 -0.          0.24455721]
 [-0.06856466 -0.66480625  0.12637421  0.09674686 -0.24455721  0.        ]]


In [8]:
print(pfA.ratio(update, ie) * newB, Pfaffian(C).evaluate())

-0.04050949817277867 -0.04050949817277863


In [9]:
pfA.accept()
print("max inverse difference: {}".format(np.max(pfA.inv - np.linalg.inv(C))))

max inverse difference: 6.946992287931846e-14


In [10]:
print(np.linalg.inv(B))

[[-3.01041281e-15  3.37758625e+00  1.51309430e+01 -7.84382153e+00
  -2.80518464e-01 -1.03781944e+01]
 [-3.37758625e+00 -1.78088739e-15 -7.96473187e+00  7.07501492e+00
  -3.69926172e-01 -1.70282163e-01]
 [-1.51309430e+01  7.96473187e+00  8.66730070e-16  9.71368287e+00
  -3.69712798e+00 -1.53008423e+01]
 [ 7.84382153e+00 -7.07501492e+00 -9.71368287e+00 -2.52522350e-15
   3.24724686e+00  1.68060062e+01]
 [ 2.80518464e-01  3.69926172e-01  3.69712798e+00 -3.24724686e+00
  -4.24653502e-16 -2.18567926e+00]
 [ 1.03781944e+01  1.70282163e-01  1.53008423e+01 -1.68060062e+01
   2.18567926e+00 -8.15626218e-15]]


# Singlet and Triplet pairing functions

In [11]:
nup = 3
ndn = 2
norbup = 6
norbdn = 6

In [12]:
up_psi = np.random.random((nup, norbup))
dn_psi = np.random.random((ndn, norbdn))

print(up_psi)
print(dn_psi)

[[0.92979771 0.04481706 0.36345255 0.55100725 0.11389361 0.83844704]
 [0.8024512  0.47750522 0.59375306 0.12799644 0.56722092 0.13476818]
 [0.81711275 0.82519555 0.78641738 0.12231961 0.52508536 0.2608769 ]]
[[0.11087974 0.91875922 0.70049105 0.1151218  0.49077789 0.36205635]
 [0.92619697 0.83850844 0.38222666 0.69600158 0.10640622 0.1246681 ]]


In [13]:
def makePairingMats(norbup, norbdn):
    S = np.random.random((norbup, norbdn))
    S = 0.5 * (S + S.T)
    
    UU = np.random.random((norbup, norbup))
    UU = 0.5 * (UU - UU.T)
    DD = np.random.random((norbdn, norbdn))
    DD = 0.5 * (DD - DD.T)
    return S, UU, DD

S, UU, DD = makePairingMats(norbup, norbdn)

In [14]:
def makePfaffianMat(up_psi, dn_psi, S, UU, DD):
    nup = up_psi.shape[0]
    ndn = dn_psi.shape[0]
    size = nup + ndn
    if size%2 != 0:
        size += 1
    pfmat = np.zeros((size, size))
    for i in range(nup + ndn):
        for j in range(i+1, nup + ndn):
            if i < nup and j < nup:
                #up up triplet
                pfmat[i,j] = np.einsum('i,ij,j->',up_psi[i,:], UU, up_psi[j,:])
            if i < nup and j >= nup:
                #up dn singlet
                pfmat[i,j] = np.einsum('i,ij,j->',up_psi[i,:],S,dn_psi[j - nup,:])
            if i >= nup and j >= nup:
                #dn dn triplet
                pfmat[i,j] = np.einsum('i,ij,j->',dn_psi[i - nup,:],DD,dn_psi[j-nup,:])
    if size == nup + ndn + 1:
        for i in range(nup):
             #unpaired
            pfmat[i,nup+ndn] = up_psi[i,i]
        for i in range(nup, nup + ndn):
            #unpaired dn
            pfmat[i, nup+ndn] = dn_psi[i-nup, i-nup]
      
    pfmat = (pfmat - pfmat.T)
    return pfmat

pfmat = makePfaffianMat(up_psi, dn_psi, S, UU, DD)
size = pfmat.shape[0]
print(pfmat)

[[ 0.          0.31465432  0.44269798  3.58456483  3.98435198  0.92979771]
 [-0.31465432  0.         -0.02471128  3.01883056  3.79901329  0.47750522]
 [-0.44269798  0.02471128  0.          3.64450499  4.60860067  0.78641738]
 [-3.58456483 -3.01883056 -3.64450499  0.          0.21021187  0.11087974]
 [-3.98435198 -3.79901329 -4.60860067 -0.21021187  0.          0.83850844]
 [-0.92979771 -0.47750522 -0.78641738 -0.11087974 -0.83850844  0.        ]]


In [15]:
for i in range(S.shape[0]):
    line = "row{} = {{ ".format(i)
    for j in range(S.shape[1]):
        if j > 0:
            line += ", "
        line += "{} ".format(S[i,j])
    line += "};"
    print(line)

row0 = { 0.3180414592166366 , 0.600104959392495 , 0.4642438313743457 , 0.5816172764398225 , 0.6037109397256399 , 0.2951113484296276 };
row1 = { 0.600104959392495 , 0.09697842311870486 , 0.41937315764161337 , 0.5421635350896339 , 0.2548558364689513 , 0.4271829154212393 };
row2 = { 0.4642438313743457 , 0.41937315764161337 , 0.6797705794844521 , 0.6307981709790462 , 0.28373386258295497 , 0.15635159921667535 };
row3 = { 0.5816172764398225 , 0.5421635350896339 , 0.6307981709790462 , 0.8418923779517383 , 0.44010957459901245 , 0.14809503193049156 };
row4 = { 0.6037109397256399 , 0.2548558364689513 , 0.28373386258295497 , 0.44010957459901245 , 0.17414766814937777 , 0.6661621789790062 };
row5 = { 0.2951113484296276 , 0.4271829154212393 , 0.15635159921667535 , 0.14809503193049156 , 0.6661621789790062 , 0.7995336027593793 };


In [16]:
for i in range(UU.shape[0]):
    line = "row{} = {{ ".format(i)
    for j in range(UU.shape[1]):
        if j > 0:
            line += ", "
        line += "{} ".format(UU[i,j])
    line += "};"
    print(line)

row0 = { 0.0 , 0.10179712049738304 , 0.04720517011113051 , 0.13767202491931702 , -0.04487759834743943 , 0.1305860299961965 };
row1 = { -0.10179712049738304 , 0.0 , -0.060719691450885904 , -0.1700387281327197 , 0.1505147613342654 , -0.1258821272345353 };
row2 = { -0.04720517011113051 , 0.060719691450885904 , 0.0 , -0.17528784342592918 , 0.18023264504075892 , -0.2124931747481953 };
row3 = { -0.13767202491931702 , 0.1700387281327197 , 0.17528784342592918 , 0.0 , 0.011885828639267404 , -0.2724282490725342 };
row4 = { 0.04487759834743943 , -0.1505147613342654 , -0.18023264504075892 , -0.011885828639267404 , 0.0 , -0.2527551842515271 };
row5 = { -0.1305860299961965 , 0.1258821272345353 , 0.2124931747481953 , 0.2724282490725342 , 0.2527551842515271 , 0.0 };


In [17]:
for i in range(DD.shape[0]):
    line = "row{} = {{ ".format(i)
    for j in range(DD.shape[1]):
        if j > 0:
            line += ", "
        line += "{} ".format(DD[i,j])
    line += "};"
    print(line)

row0 = { 0.0 , -0.2841971014610199 , -0.22729801265851418 , 0.02689802638369143 , -0.014699398554927245 , -0.2408488597164405 };
row1 = { 0.2841971014610199 , 0.0 , -0.11422553793697526 , -0.09290196456215388 , 0.0845644076456245 , -0.004046219453199551 };
row2 = { 0.22729801265851418 , 0.11422553793697526 , 0.0 , 0.18361494551113666 , 0.04352167620130676 , 0.19876007777434612 };
row3 = { -0.02689802638369143 , 0.09290196456215388 , -0.18361494551113666 , 0.0 , 0.4455646986311197 , 0.42441739601566725 };
row4 = { 0.014699398554927245 , -0.0845644076456245 , -0.04352167620130676 , -0.4455646986311197 , 0.0 , 0.18807375032492452 };
row5 = { 0.2408488597164405 , 0.004046219453199551 , -0.19876007777434612 , -0.42441739601566725 , -0.18807375032492452 , 0.0 };


In [18]:
print(pfmat)
pf = Pfaffian(pfmat)

[[ 0.          0.31465432  0.44269798  3.58456483  3.98435198  0.92979771]
 [-0.31465432  0.         -0.02471128  3.01883056  3.79901329  0.47750522]
 [-0.44269798  0.02471128  0.          3.64450499  4.60860067  0.78641738]
 [-3.58456483 -3.01883056 -3.64450499  0.          0.21021187  0.11087974]
 [-3.98435198 -3.79901329 -4.60860067 -0.21021187  0.          0.83850844]
 [-0.92979771 -0.47750522 -0.78641738 -0.11087974 -0.83850844  0.        ]]


In [19]:
pf.evaluate()

np.float64(-0.5520438346362099)

# try to move a particle

In [20]:
orb_update = np.random.random(norbup)
row_update = np.zeros(size)
act = 2
for i in range(nup+ndn):
    sign = 1
    if i < act:
        sign *= -1
    if i == act:
        continue
    if i < nup and act < nup:
        row_update[i] = sign * np.einsum('i,ij,j->',orb_update, UU, up_psi[i,:])
    if i < nup and act >= nup:
        row_update[i] = sign * np.einsum('i,ij,j->', orb_update, S, dn_psi[i-nup,:])
    if i >= nup and act >= ndn:
        row_update[i] = sign * np.einsum('i,ij,j->', orb_update, DD, dn_psi[i-nup,:])
if size == nup + ndn + 1:
    if act < nup:
        row_update[-1] = orb_update[act]
    if act >= ndn:
        row_update[-1] = orb_update[act - nup]
print(orb_update)
print(row_update)

[0.26400195 0.83715123 0.13608529 0.41801178 0.39974942 0.4264682 ]
[ 0.39532238 -0.02721039  0.         -0.12400164 -0.0691121   0.4264682 ]


In [21]:
newmat = pfmat.copy()
newmat[act,:] = row_update
newmat[:,act] = -row_update
print(row_update)
print(pfmat)
print(newmat)

[ 0.39532238 -0.02721039  0.         -0.12400164 -0.0691121   0.4264682 ]
[[ 0.          0.31465432  0.44269798  3.58456483  3.98435198  0.92979771]
 [-0.31465432  0.         -0.02471128  3.01883056  3.79901329  0.47750522]
 [-0.44269798  0.02471128  0.          3.64450499  4.60860067  0.78641738]
 [-3.58456483 -3.01883056 -3.64450499  0.          0.21021187  0.11087974]
 [-3.98435198 -3.79901329 -4.60860067 -0.21021187  0.          0.83850844]
 [-0.92979771 -0.47750522 -0.78641738 -0.11087974 -0.83850844  0.        ]]
[[ 0.          0.31465432 -0.39532238  3.58456483  3.98435198  0.92979771]
 [-0.31465432  0.          0.02721039  3.01883056  3.79901329  0.47750522]
 [ 0.39532238 -0.02721039 -0.         -0.12400164 -0.0691121   0.4264682 ]
 [-3.58456483 -3.01883056  0.12400164  0.          0.21021187  0.11087974]
 [-3.98435198 -3.79901329  0.0691121  -0.21021187  0.          0.83850844]
 [-0.92979771 -0.47750522 -0.4264682  -0.11087974 -0.83850844  0.        ]]


In [22]:
newpf = Pfaffian(newmat)

In [23]:
print(newpf.evaluate())

0.14246409452636136


In [24]:
print(pf.evaluate() * pf.ratio(row_update, act))

0.14246409452636027


# check gradients and laplacians for actual orbital

In [25]:
def makeOrbs(elec_pos, up_basis, dn_basis, nup, ndn):
    psi_up = np.cos(elec_pos[:nup,:] @ up_basis.T)
    dpsi_up = np.zeros((nup, norbup, 3))
    d2psi_up = np.zeros((nup, norbup))
    for ie in range(nup):
        for io in range(norbup):
            for d in range(3):
                dpsi_up[ie, io, d] = -up_basis[io, d] * np.sin(elec_pos[ie,:].dot(up_basis[io,:]))
            d2psi_up[ie, io] = -up_basis[io,:].dot(up_basis[io,:]) * psi_up[ie, io]
    psi_dn = np.cos(elec_pos[nup:nup+ndn,:] @ dn_basis.T)
    dpsi_dn = np.zeros((ndn, norbdn, 3))
    d2psi_dn = np.zeros((ndn, norbdn))
    for ie in range(ndn):
        for io in range(norbdn):
            for d in range(3):
                dpsi_dn[ie, io, d] = -dn_basis[io, d] * np.sin(elec_pos[nup+ie,:].dot(dn_basis[io,:]))
            d2psi_dn[ie, io] = -dn_basis[io,:].dot(dn_basis[io,:]) * psi_dn[ie, io]
    return psi_up, dpsi_up, d2psi_up, psi_dn, dpsi_dn, d2psi_dn

In [26]:
def makePfaffianDerivMats(psi_up, dpsi_up, d2psi_up,
                       psi_dn, dpsi_dn, d2psi_dn,
                       S, UU, DD,
                       iat):
    nup = psi_up.shape[0]
    ndn = psi_dn.shape[0]
    ne  = nup + ndn

    size  = ne + (ne % 2)
    dA  = [ np.zeros((size,size)) for _ in range(3) ]
    # the sum over d of second derivatives
    d2A = np.zeros((size,size))
    
    def blk_idx(i):
        if i < nup:
            return 'up', i
        else:
            return 'dn', i - nup

    for i in range(ne):
        il, ii = blk_idx(i)
        for j in range(i+1, ne):
            jl, jj = blk_idx(j)
            if  il == 'up' and jl == 'up':
                mat            = UU
                psi_i, psi_j   = psi_up[ii,:], psi_up[jj,:]
                dpsi_i, dpsi_j = dpsi_up[ii,:,:],   dpsi_up[jj,:,:]
                d2psi_i, d2psi_j = d2psi_up[ii,:], d2psi_up[jj,:]
            elif il=='up' and jl=='dn':
                mat            = S
                psi_i, psi_j   = psi_up[ii,:], psi_dn[jj,:]
                dpsi_i, dpsi_j = dpsi_up[ii,:,:],   dpsi_dn[jj,:,:]
                d2psi_i, d2psi_j = d2psi_up[ii,:], d2psi_dn[jj,:]
            else:  # dn, dn
                mat            = DD
                psi_i, psi_j   = psi_dn[ii,:], psi_dn[jj,:]
                dpsi_i, dpsi_j = dpsi_dn[ii,:,:],   dpsi_dn[jj,:,:]
                d2psi_i, d2psi_j = d2psi_dn[ii,:], d2psi_dn[jj,:]
            
            for d in range(3):
                if i == iat:                    
                    dA[d][i,j] = np.einsum('i,ij,j->', dpsi_i[:,d], mat, psi_j)
                elif j == iat:
                    dA[d][i,j] = np.einsum('i,ij,j->', psi_i, mat, dpsi_j[:,d])
                    
            # summed‐over‐d second derivative ("laplacian" of this matrix element)
            if i == iat:
                d2A[i,j] = np.einsum('i,ij,j->', d2psi_i, mat, psi_j)
            elif j == iat:
                d2A[i,j] = np.einsum('i,ij,j->', psi_i, mat, d2psi_j)

    if size == nup + ndn + 1:
        for i in range(ne):
            il, ii = blk_idx(i)
            if il=='up' and ii < nup:
                psi_i    = psi_up[ii,ii]
                dpsi_i   = dpsi_up[ii, ii,:] 
                d2psi_i  = d2psi_up[ii,ii]  
            elif il=='dn' and ii < ndn:
                psi_i    = psi_dn[ii,ii]
                dpsi_i   = dpsi_dn[ii, ii,:]
                d2psi_i  = d2psi_dn[ii,ii]   
            else:
                psi_i    = 0
                dpsi_i   = np.zeros(3)
                d2psi_i  = 0  

            for d in range(3):
                if i == iat:
                    dA[d][i,ne] = dpsi_i[d]
            # summed second
            if i == iat:
                d2A[i,ne]  = d2psi_i

    # antisymmetrize them
    for d in range(3):
        dA[d] = dA[d]  - dA[d].T
    d2A = d2A - d2A.T

    return dA, d2A

In [27]:
elec_pos = np.random.random((nup+ndn, 3))
up_basis = np.random.random((norbup, 3))
dn_basis = np.random.random((norbdn, 3))

In [28]:
def finiteDifferences(ref_elec, up_basis, dn_basis, S, UU, DD, iat, dr = 1e-3):
    psi_up, dpsi_up, d2psi_up, psi_dn, dpsi_dn, d2psi_dn = makeOrbs(ref_elec, up_basis, dn_basis, nup, ndn)
    refmat = makePfaffianMat(psi_up, psi_dn, S, UU, DD)
    ref = Pfaffian(refmat).evaluate()
    grad = np.zeros(3)
    d2 = np.zeros(3)
    for d in range(grad.shape[0]):
        new_elec = ref_elec.copy()
        new_elec[iat, d] -= dr
        psi_up, dpsi_up, d2psi_up, psi_dn, dpsi_dn, d2psi_dn = makeOrbs(new_elec, up_basis, dn_basis, nup, ndn)
        minus = makePfaffianMat(psi_up, psi_dn, S, UU, DD)
        pfm = Pfaffian(minus).evaluate()
        
        new_elec[iat, d] += 2*dr
        psi_up, dpsi_up, d2psi_up, psi_dn, dpsi_dn, d2psi_dn = makeOrbs(new_elec, up_basis, dn_basis, nup, ndn)
        plus = makePfaffianMat(psi_up, psi_dn, S, UU, DD)
        pfp = Pfaffian(plus).evaluate()
        
        grad[d] = (pfp - pfm)/(2 * dr)
        d2[d] = (pfp - 2 * ref + pfm) / dr**2
    lap = d2.sum()

    grad_log_psi = grad/ref
    lap_log_psi = lap/ref - grad_log_psi.dot(grad_log_psi)
    print(grad_log_psi, lap_log_psi)

In [29]:

psi_up, dpsi_up, d2psi_up, psi_dn, dpsi_dn, d2psi_dn = makeOrbs(elec_pos, up_basis, dn_basis, nup, ndn)
refmat = makePfaffianMat(psi_up, psi_dn, S, UU, DD)

print("up SPO VGL")
print(psi_up)
print(dpsi_up)
print(d2psi_up)
print()

print("dn SPO VGL")
print(psi_dn)
print(dpsi_dn)
print(d2psi_dn)
print()

pf = Pfaffian(refmat)

print("pfaffian value = {}\n\n".format(pf.evaluate()))

for ie in range(nup + ndn):
    print("particle {}".format(ie))
    print("finite differences")
    finiteDifferences(elec_pos, up_basis, dn_basis, S, UU, DD, ie)
    dA, d2A = makePfaffianDerivMats(psi_up, dpsi_up, d2psi_up,
                                    psi_dn, dpsi_dn, d2psi_dn,
                                    S, UU, DD,
                                    ie)
    gradlogpsi = np.zeros(3)
    dot = 0.0
    for d in range(3):
        gradlogpsi[d] = 0.5 * np.trace(np.matmul(pf.inv, dA[d]))
        dot += 0.5 * np.trace(np.matmul(np.matmul(pf.inv, dA[d]), np.matmul(pf.inv, dA[d])))
    d2logpsi = 0.5 * np.trace(np.matmul(pf.inv, d2A)) - dot

        
    print("trace formula")
    print(gradlogpsi, d2logpsi)
    print("\n\n")   

up SPO VGL
[[ 0.26880981  0.38297036  0.67970534  0.57913668  0.34154767  0.68877715]
 [-0.00249844  0.07922684  0.51539003  0.39623433  0.10879198  0.5439395 ]
 [ 0.38216715  0.31236072  0.70069793  0.55406334  0.36763894  0.64983887]]
[[[-0.64400252 -0.91610297 -0.48422924]
  [-0.44093363 -0.45549765 -0.77629632]
  [-0.37069611 -0.38923311 -0.23519604]
  [-0.13093672 -0.33648282 -0.6720715 ]
  [-0.19117277 -0.61472653 -0.90141821]
  [-0.06633773 -0.21230168 -0.56491323]]

 [[-0.66860985 -0.95110725 -0.50273163]
  [-0.47582413 -0.49154059 -0.83772365]
  [-0.43309684 -0.45475425 -0.27478751]
  [-0.14746686 -0.37896217 -0.75691732]
  [-0.20219732 -0.65017657 -0.95340118]
  [-0.076783   -0.24572984 -0.65386216]]

 [[-0.61785976 -0.8789145  -0.46457234]
  [-0.45344093 -0.46841808 -0.79831636]
  [-0.36057456 -0.37860542 -0.22877421]
  [-0.13370634 -0.34360023 -0.6862874 ]
  [-0.18915991 -0.60825407 -0.89192716]
  [-0.0695494  -0.22258006 -0.59226296]]]
[[-0.43127665 -0.45082918 -0.43490375